In [0]:
%pip install -q torch scikit-learn





In [0]:
%run ../utils/utils

# Treinamento — Autoencoder para Anomalias de Vendas (squad1 + squad3)

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)

CAMINHO_MODELO = "IA/encouders/autoencoder_pytorch.pt"
CAMINHO_SCALER = "IA/encouders/scaler.joblib"
CAMINHO_METADADOS = "IA/encouders/metadados.json"


##  Ler a união já pronta

In [0]:
df_pedidos_unificado = ler_delta("IA/encouders", "stg_pedidos_unificado", STORAGE_OPTIONS)
df_itens_unificado = ler_delta("IA/encouders", "stg_itens_unificado", STORAGE_OPTIONS)
df_produtos_unificado = ler_delta("IA/encouders", "stg_produtos_unificado", STORAGE_OPTIONS)

print(f"Pedidos unificados (lidos da Etapa 1): {df_pedidos_unificado.count()}")


## Construir features a partir dos dados unificados

In [0]:
df_features = construir_features_pedidos(
    df_pedidos_unificado,
    df_itens_unificado,
    df_produtos_unificado
)
print(f"Total de pedidos (squad1 + squad3) para o modelo: {df_features.count()}")

## Split temporal (80/20)

In [0]:
PERCENTUAL_TREINO = 0.80


df_com_data = df_pedidos_unificado.select("id_pedido", "dt_pedido", "origem_squad").join(df_features, "id_pedido")
df_com_indice = df_com_data.withColumn(
    "indice_temporal", F.row_number().over(Window.orderBy("dt_pedido"))
)

qtd_total = df_com_indice.count()
qtd_treino = int(qtd_total * PERCENTUAL_TREINO)

df_treino_spark = df_com_indice.filter(F.col("indice_temporal") <= qtd_treino).drop("indice_temporal", "dt_pedido")
df_teste_spark = df_com_indice.filter(F.col("indice_temporal") > qtd_treino).drop("indice_temporal", "dt_pedido")

print(f"Treino: {df_treino_spark.count()} pedidos | Teste: {df_teste_spark.count()} pedidos")


## Encoding e escala


In [0]:
pdf_treino = df_treino_spark.toPandas()
pdf_teste = df_teste_spark.toPandas()

pdf_treino_encoded = pd.get_dummies(pdf_treino, columns=["metodo_pagamento"], prefix="pgto")
colunas_onehot = sorted([c for c in pdf_treino_encoded.columns if c.startswith("pgto_")])
colunas_features = COLUNAS_NUMERICAS_ANOMALIA + colunas_onehot

pdf_teste_encoded = pd.get_dummies(pdf_teste, columns=["metodo_pagamento"], prefix="pgto")

X_treino = pdf_treino_encoded.reindex(columns=colunas_features, fill_value=0).fillna(0).values.astype("float32")
X_teste = pdf_teste_encoded.reindex(columns=colunas_features, fill_value=0).fillna(0).values.astype("float32")

scaler = StandardScaler()
X_treino_scaled = scaler.fit_transform(X_treino).astype("float32")
X_teste_scaled = scaler.transform(X_teste).astype("float32")

qtd_features = X_treino_scaled.shape[1]
print(f"Dimensão de entrada: {qtd_features} features")

## Arquitetura do Autoencoder

In [0]:
dim_entrada = qtd_features
dim_oculta1 = max(8, dim_entrada // 2)
dim_gargalo = max(3, dim_entrada // 4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
autoencoder = AutoencoderTorch(dim_entrada, dim_oculta1, dim_gargalo).to(device)
print(autoencoder)

## Treinamento


In [0]:
n_val = int(len(X_treino_scaled) * 0.15)
indices = np.random.permutation(len(X_treino_scaled))
idx_val, idx_treino_real = indices[:n_val], indices[n_val:]

X_treino_real = torch.tensor(X_treino_scaled[idx_treino_real]).to(device)
X_val = torch.tensor(X_treino_scaled[idx_val]).to(device)

otimizador = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
funcao_perda = nn.MSELoss()

EPOCAS, BATCH_SIZE, PATIENCE = 100, 64, 10
melhor_val_loss = float("inf")
epocas_sem_melhora = 0
melhores_pesos = None
historico_loss_treino, historico_val_loss = [], []

dataset_treino = torch.utils.data.TensorDataset(X_treino_real, X_treino_real)
loader_treino = torch.utils.data.DataLoader(dataset_treino, batch_size=BATCH_SIZE, shuffle=True)

for epoca in range(EPOCAS):
    autoencoder.train()
    perda_acumulada = 0.0
    for batch_x, batch_y in loader_treino:
        otimizador.zero_grad()
        reconstrucao = autoencoder(batch_x)
        perda = funcao_perda(reconstrucao, batch_y)
        perda.backward()
        otimizador.step()
        perda_acumulada += perda.item() * batch_x.size(0)

    loss_treino = perda_acumulada / len(dataset_treino)

    autoencoder.eval()
    with torch.no_grad():
        reconstrucao_val = autoencoder(X_val)
        val_loss = funcao_perda(reconstrucao_val, X_val).item()

    historico_loss_treino.append(loss_treino)
    historico_val_loss.append(val_loss)

    print(f"Epoch {epoca+1}/{EPOCAS} - loss: {loss_treino:.4f} - val_loss: {val_loss:.4f}")

    if val_loss < melhor_val_loss:
        melhor_val_loss = val_loss
        epocas_sem_melhora = 0
        melhores_pesos = {k: v.clone() for k, v in autoencoder.state_dict().items()}
    else:
        epocas_sem_melhora += 1
        if epocas_sem_melhora >= PATIENCE:
            print(f"EarlyStopping: sem melhora por {PATIENCE} épocas seguidas. Parando.")
            break

# Restaura os melhores pesos vistos (equivalente a restore_best_weights=True)
autoencoder.load_state_dict(melhores_pesos)

## Limiar de anomalia (percentil 95 do erro de treino)

In [0]:
PERCENTIL_LIMIAR = 95

autoencoder.eval()
with torch.no_grad():
    reconstrucao_treino = autoencoder(torch.tensor(X_treino_scaled).to(device)).cpu().numpy()
    reconstrucao_teste = autoencoder(torch.tensor(X_teste_scaled).to(device)).cpu().numpy()

erro_treino = np.mean(np.square(X_treino_scaled - reconstrucao_treino), axis=1)
erro_teste = np.mean(np.square(X_teste_scaled - reconstrucao_teste), axis=1)
limiar_anomalia = float(np.percentile(erro_treino, PERCENTIL_LIMIAR))

pdf_teste["erro_reconstrucao"] = erro_teste
pdf_teste["is_anomaly"] = erro_teste > limiar_anomalia

print(f"Limiar (p{PERCENTIL_LIMIAR}): {limiar_anomalia:.4f}")
print(f"Anomalias no teste: {pdf_teste['is_anomaly'].sum()} de {len(pdf_teste)}")

##  Auditoria no conjunto de teste

In [0]:
display(
    pdf_teste.sort_values("erro_reconstrucao", ascending=False)
    .head(20)[["id_pedido", "id_cliente", "valor_total", "desvio_pct_vs_media_cliente",
               "hora_do_dia", "qtd_itens", "erro_reconstrucao"]]
)

## Visualização dos resultados

In [0]:
import matplotlib.pyplot as plt

fig, eixos = plt.subplots(1, 3, figsize=(18, 5))

# 1. Curva de aprendizado (loss de treino vs validação por época)
eixos[0].plot(historico_loss_treino, label="loss (treino)")
eixos[0].plot(historico_val_loss, label="val_loss (validação)")
eixos[0].set_title("Curva de aprendizado")
eixos[0].set_xlabel("Época")
eixos[0].set_ylabel("MSE")
eixos[0].legend()

# 2. Distribuição do erro de reconstrução no TESTE, com o limiar marcado
eixos[1].hist(pdf_teste["erro_reconstrucao"], bins=40, color="#4C72B0", alpha=0.8)
eixos[1].axvline(limiar_anomalia, color="red", linestyle="--", label=f"Limiar (p{PERCENTIL_LIMIAR})")
eixos[1].set_title("Distribuição do erro de reconstrução (teste)")
eixos[1].set_xlabel("Erro de reconstrução")
eixos[1].set_ylabel("Qtd. de pedidos")
eixos[1].legend()

# 3. Contagem normal vs anomalia no teste
contagem = pdf_teste["is_anomaly"].value_counts().reindex([False, True], fill_value=0)
eixos[2].bar(["Normal", "Anomalia"], contagem.values, color=["#4C72B0", "#C44E52"])
eixos[2].set_title(f"Pedidos classificados no teste (n={len(pdf_teste)})")
eixos[2].set_ylabel("Qtd. de pedidos")
for i, v in enumerate(contagem.values):
    eixos[2].text(i, v, str(v), ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()

print(f"\nResumo: {contagem[True]} anomalias detectadas de {len(pdf_teste)} pedidos no conjunto de teste "
      f"({100 * contagem[True] / len(pdf_teste):.2f}%).")
print(f"Melhor val_loss atingida: {melhor_val_loss:.4f} (parada na época {len(historico_loss_treino)}).")

## Salvar modelo, scaler, metadados, histórico de loss e previsões do teste

In [0]:
import json

# 8.1 Modelo, scaler, metadados
salvar_modelo_pytorch(autoencoder, CAMINHO_MODELO, container_squad1)
salvar_modelo_ml(scaler, CAMINHO_SCALER, container_squad1)

metadados = {
    "colunas_features": colunas_features,
    "limiar_anomalia": limiar_anomalia,
    "percentil_limiar": PERCENTIL_LIMIAR,
    "fontes_dados": ["squad1", "squad3"],
    "dim_entrada": dim_entrada,
    "dim_oculta1": dim_oculta1,
    "dim_gargalo": dim_gargalo,
    "melhor_val_loss": melhor_val_loss,
    "qtd_epocas_treinadas": len(historico_loss_treino),
}
file_client = container_squad1.get_file_client(CAMINHO_METADADOS)
file_client.upload_data(json.dumps(metadados, ensure_ascii=False, indent=2), overwrite=True)

# 8.2 Histórico de loss (para o gráfico da curva de aprendizado na análise)
df_historico_loss = spark.createDataFrame(
    [(i + 1, historico_loss_treino[i], historico_val_loss[i]) for i in range(len(historico_loss_treino))],
    ["epoca", "loss_treino", "val_loss"]
)
gravar_delta(df=df_historico_loss, camada="IA/encouders", tabela="stg_treino_historico_loss",
             storage_opts=STORAGE_OPTIONS, mode="overwrite", particionar=False)

# 8.3 Previsões do conjunto de teste, JÁ com origem_squad (para a análise de
# desbalanceamento entre squads) — assim a análise nem precisa reler
# df_pedidos_unificado.
colunas_para_salvar = ["id_pedido", "id_cliente", "origem_squad", "valor_total",
                        "desvio_pct_vs_media_cliente", "hora_do_dia", "qtd_itens",
                        "erro_reconstrucao", "is_anomaly"] + COLUNAS_NUMERICAS_ANOMALIA
colunas_para_salvar = list(dict.fromkeys(colunas_para_salvar))  # remove duplicatas mantendo ordem

df_teste_resultado = spark.createDataFrame(pdf_teste[colunas_para_salvar])
gravar_delta(df=df_teste_resultado, camada="IA/encouders", tabela="stg_teste_predicoes",
             storage_opts=STORAGE_OPTIONS, mode="overwrite", particionar=False)

print("Modelo, scaler, metadados, histórico de loss e previsões do teste salvos.")
print("A Etapa 3 (análise) agora só precisa LER essas tabelas, sem retreinar nada.")
